In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, BatchNormalization
import numpy as np

In [ ]:
# Load the dataset and tokenize the text (assuming 'text' contains the sonnet text)
# Load the dataset
with open('Sonnet.txt', 'r') as file:
    text = file.read()


# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:




# Create input sequences
input_sequences = []
sonnets = text.split('\n\n\n')  # Split text into sonnets

for sonnet in sonnets:
    lines = sonnet.split('\n')  # Split sonnet into lines
    for line in lines:
        if line.strip():  # Ensure the line is not empty
            token_list = tokenizer.texts_to_sequences([line])[0]
            for i in range(1, len(token_list)):
                n_gram_sequence = token_list[:i+1]
                input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# Create predictors and label
input_sequences = np.array(input_sequences)
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Define the modified LSTM model
model = Sequential()
model.add(Embedding(total_words, 256, input_length=max_sequence_len-1))
model.add(Bidirectional(LSTM(512, return_sequences=True)))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(512)))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with the modified architecture
history = model.fit(X, y, epochs=100, batch_size=128, validation_split=0.2)

# Save the entire model in TensorFlow SavedModel format
model.save('shakespeare_sonnet_lstm_model', save_format='tf')

# Evaluate the model and check the performance
# (You can use the validation set for evaluation)
loss, accuracy = model.evaluate(X_val, y_val)  # Assuming X_val and y_val are your validation data
print(f'Validation Loss: {loss}')
print(f'Validation Accuracy: {accuracy}')



Epoch 1/100
97/97 [==============================] - 287s 3s/step - loss: 7.8735 - accuracy: 0.0225 - val_loss: 7.2644 - val_accuracy: 0.0291
Epoch 2/100
97/97 [==============================] - 274s 3s/step - loss: 6.6920 - accuracy: 0.0450 - val_loss: 7.0437 - val_accuracy: 0.0275
Epoch 3/100
97/97 [==============================] - 278s 3s/step - loss: 5.9335 - accuracy: 0.0736 - val_loss: 7.1603 - val_accuracy: 0.0297
Epoch 4/100
97/97 [==============================] - 279s 3s/step - loss: 5.2008 - accuracy: 0.1185 - val_loss: 7.2110 - val_accuracy: 0.0375
Epoch 5/100
97/97 [==============================] - 267s 3s/step - loss: 4.3007 - accuracy: 0.1925 - val_loss: 7.3144 - val_accuracy: 0.0378
Epoch 6/100
97/97 [==============================] - 273s 3s/step - loss: 3.2912 - accuracy: 0.3188 - val_loss: 7.6299 - val_accuracy: 0.0372
Epoch 7/100
97/97 [==============================] - 274s 3s/step - loss: 2.3782 - accuracy: 0.4845 - val_loss: 7.9867 - val_accuracy: 0.0420
Epoch 